<a href="https://colab.research.google.com/github/IAT-ExploringAI-2024/Computer-Vision-Project-Overview/blob/main/Copy_of_YOLOv8_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Repository
Master dataset can be found here:
https://drive.google.com/drive/folders/1BuG1DCrTehLr9O0lKzYHJlbgGkdfKsMZ?usp=drive_link

# Setup

Pip install `ultralytics` and [dependencies](https://github.com/ultralytics/ultralytics/blob/main/requirements.txt) and check software and hardware. Set up Google Drive as well.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

# 1. Predict / Dependency Check

YOLOv8 may be used directly in the Command Line Interface (CLI) with a `yolo` command for a variety of tasks and modes and accepts additional arguments, i.e. `imgsz=640`. See a full list of available `yolo` [arguments](https://docs.ultralytics.com/usage/cfg/) and other details in the [YOLOv8 Predict Docs](https://docs.ultralytics.com/modes/train/).


if CLI the format should be:

    yolo TASK MODE ARGS

  Where:

    TASK (optional) is one of (detect, segment, classify, pose)


    MODE (required) is one of (train, val, predict, export, track)


    ARGS (optional) are arg=value pairs like imgsz=640 that override defaults.


Default ARG values are defined on this page from the cfg/defaults.yaml file.



In [ ]:
# Run inference on an image with YOLOv8n

# Detection:
!yolo predict model=yolov8n.pt source='https://ultralytics.com/images/zidane.jpg'
!yolo predict model=yolov8n.pt source='https://www.thoughtco.com/thmb/Hw83C2QW8dWeDUt7R4pYyPke3HI=/1500x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/GettyImages-547031277-58ef97803df78cd3fc724e24.jpg' save=True
!yolo predict model=yolov8n.pt source=0

# Segmentation:
!yolo segment predict  model=yolov8s-seg.pt source='https://ultralytics.com/images/zidane.jpg'

#pose detection:
!yolo classify predict  model=yolov8s-cls.pt source='https://ultralytics.com/images/zidane.jpg'


#add arguments of "save and show"

In [ ]:
# SHOW THE IMAGE STORED
%matplotlib inline
from PIL import Image
test_img_path = "/content/drive/MyDrive/SFU Stuff/IAT 360/CV Project/Notebooks/Sandbox/master-data/test/images/000012.jpg"
Image.open(test_img_path)

you can also use functions of ultralytics intead of CLI commands.

In [ ]:
%matplotlib inline
from PIL import Image

from ultralytics import YOLO

# Load a pretrained YOLOv8n model
model = YOLO('yolov8n.pt')

# Run inference on test image (truck)
results = model(test_img_path)  # results list

# Show the results
for r in results:
    im_array = r.plot()  # plot a BGR numpy array of predictions
    im = Image.fromarray(im_array[..., ::-1])  # RGB PIL image
    im.show()  # show (doesn't work on colab)
    im.save('results.jpg')  # save image


#show image directly
from google.colab.patches import cv2_imshow
cv2_imshow(im_array)

#show saved image
#Image.open('results.jpg')

# 2. Val / Dependency Check pt 2
Validate a model's accuracy on the [COCO](https://docs.ultralytics.com/datasets/detect/coco/) dataset's `val` or `test` splits. The latest YOLOv8 [models](https://github.com/ultralytics/ultralytics#models) are downloaded automatically the first time they are used. See [YOLOv8 Val Docs](https://docs.ultralytics.com/modes/val/) for more information.

In [ ]:
# Download COCO val
import torch
torch.hub.download_url_to_file('https://ultralytics.com/assets/coco2017val.zip', 'tmp.zip')  # download (780M - 5000 images)
!unzip -q tmp.zip -d datasets && rm tmp.zip  # unzip

In [ ]:
# Validate YOLOv8n on COCO8 val
!yolo val model=yolov8n.pt data=coco8.yaml

In [ ]:
#@title Select YOLOv8 🚀 logger {run: 'auto'}
logger = 'Comet' #@param ['Comet', 'TensorBoard']

if logger == 'Comet':
  %pip install -q comet_ml
  import comet_ml; comet_ml.init()
elif logger == 'TensorBoard':
  %load_ext tensorboard
  %tensorboard --logdir .


In YOLO (You Only Look Once), a YAML file is used for configuration and setup. It specifies parameters such as paths to datasets, model architecture, training hyperparameters, and class names. The YAML file is essential for defining how the YOLO model should be trained and what it should detect, making it an integral part of customizing the YOLO model for specific object detection tasks.

In code below, coco8.yaml is used:



    # Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
    path: ../datasets/coco8  # dataset root dir
    train: images/train  # train images (relative to 'path') 4 images
    val: images/val  # val images (relative to 'path') 4 images
    test:  # test images (optional)

    # Classes
    names:
      0: person
      1: bicycle
      2: car
      ...
      79: toothbrush


    # Download script/URL (optional)
    download: https://ultralytics.com/assets/coco8.zip

In [ ]:
# Train YOLOv8n on COCO8 for 3 epochs
#!yolo train model=yolov8n.pt data=coco8.yaml epochs=3 imgsz=640

!yolo train model=yolov8n.pt data=coco8-seg.yaml epochs=3 imgsz=640



# 3. Python Usage / Training

YOLOv8 was reimagined using Python-first principles for the most seamless Python YOLO experience yet. YOLOv8 models can be loaded from a trained checkpoint or created from scratch. Then methods are used to train, val, predict, and export the model. See detailed Python usage examples in the [YOLOv8 Python Docs](https://docs.ultralytics.com/usage/python/).

In [ ]:
from ultralytics import YOLO

yaml_path = "/content/drive/MyDrive/SFU Stuff/IAT 360/CV Project/Notebooks/Sandbox/master-data/data.yaml"
checkpoint = "/content/runs/detect/train/weights/last.pt"

# Load a model
# model = YOLO('yolov8n.yaml')  # build a new model from scratch
model = YOLO(checkpoint)  # load a pretrained model (recommended for training)

# Use the model
results = model.train(data=yaml_path, epochs=30, patience=10, hsv_h=0.03, hsv_s=0.6, hsv_v=0.5, optimizer="sgd")  # train the model
results = model.val()  # evaluate model performance on the validation set
results = model("/content/drive/MyDrive/SFU Stuff/IAT 360/CV Project/Notebooks/Sandbox/master-data/test/images/000012.jpg")  # predict on an image
#results = model.export(format='onnx')  # export the model to ONNX format

In [ ]:
# Export model
results = model.export(format='onnx')  # export the model to ONNX format

## 1. Detection

YOLOv8 _detection_ models have no suffix and are the default YOLOv8 models, i.e. `yolov8n.pt` and are pretrained on COCO. See [Detection Docs](https://docs.ultralytics.com/tasks/detect/) for full details.


In [ ]:
# Load YOLOv8n, train it on COCO128 for 300 epochs and predict an image with it
from ultralytics import YOLO

model = YOLO('/content/drive/MyDrive/Colab Notebooks/runs/detect/train4/weights/best.pt')  # load a pretrained YOLOv8n detection model
# model.train(data=yaml_path, epochs=10)  # train the model
# model('/content/drive/MyDrive/SFU Stuff/IAT 360/CV Project/Data/Sample Dataset/Tim Set/cleaned_v2/17.jpg')  # predict on an image

### Logged Metrics

In [ ]:
model= YOLO("/content/drive/MyDrive/Colab Notebooks/runs/detect/train4/weights/best.pt")
metrics = model.val()  # no arguments needed, dataset and settings remembered
print(metrics)

In [ ]:
results=model('https://miro.medium.com/v2/resize:fit:510/format:webp/1*jW-Q9DvmB-zvM4hgJlBx3g.png', save=True)
#results
for r in results:
    print(r.probs)  # print the Probs object containing the detected class probabilities


# **Results / Findings**

Some preliminary results and findings so far from runs directory after epoch=3

#  **Findings (Tim) - Epoch=3**

Findings of model after 3 epochs and AdamW optimizer

## Group A - Prediction Samples

In [ ]:
from PIL import Image
predict_img_path = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val/val_batch0_pred.jpg"
img = Image.open(predict_img_path)
img.resize((1080, 720))

## Group A - Labels (Groundtruths) Samples

In [ ]:
from PIL import Image
truth_img_path = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val/val_batch0_labels.jpg"
img = Image.open(truth_img_path)
img.resize((1080, 720))

## Group A - Graphs

### Confusion Matrix

In [ ]:
from PIL import Image
matrix_img_path = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val/confusion_matrix.png"
img = Image.open(matrix_img_path)
img.resize((980, 720))

### Confusion Matrix: Normalized

In [ ]:
from PIL import Image
norm_matrix_img_path = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val/confusion_matrix_normalized.png"
img = Image.open(norm_matrix_img_path)
img.resize((980, 720))

### F1-Curve

In [ ]:
from PIL import Image
f1_curve = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val/BoxF1_curve.png"
img = Image.open(f1_curve)
img.resize((980, 720))

# **Findings (Tim) - Epoch=90**

Findings of model after 90 epochs and SGD optimizer.

Learning plateaued around epoch=60 with an mAP50-95 of ~0.52

Training was done with SGD with lr=0.01, momentum=0.9 and imgsz of 640.

## Sample B - Prediction Samples

In [ ]:
from PIL import Image
predict_img_path = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val3/val_batch0_pred.jpg"
img = Image.open(predict_img_path)
img.resize((1080, 720))

## Group B - Labels (Groundtruths) Samples

In [ ]:
from PIL import Image
truth_img_path = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val3/val_batch0_labels.jpg"
img = Image.open(truth_img_path)
img.resize((1080, 720))

## Group B - Graphs

### Confusion Matrix

In [ ]:
from PIL import Image
matrix_img_path = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val3/confusion_matrix.png"
img = Image.open(matrix_img_path)
img.resize((980, 720))

### Confusion Matrix: Normalized

In [ ]:
from PIL import Image
norm_matrix_img_path = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val3/confusion_matrix_normalized.png"
img = Image.open(norm_matrix_img_path)
img.resize((980, 720))

### F1 Curve

In [ ]:
from PIL import Image
f1_curve = "/content/drive/MyDrive/Colab Notebooks/runs/detect/val3/BoxF1_curve.png"
img = Image.open(f1_curve)
img.resize((980, 720))

# **Test Set (Tim)**
Quick test run of sample data set.

## Sample Dataset Setup

### Test Load (Sanity Check)

In [ ]:
# Load best model
model= YOLO("/content/drive/MyDrive/Colab Notebooks/runs/detect/train4/weights/best.pt")

# Predict some image
results = model.predict("/content/drive/MyDrive/SFU Stuff/IAT 360/CV Project/Data/Sample Dataset/Tim Set/cleaned_v2/17.jpg")

### Predict and display bounding boxes for all images in dataset.

In [ ]:
from ultralytics import YOLO
import cv2
import glob
import os
from google.colab.patches import cv2_imshow  # use cv2.imshow() if local

# Load trained model
model_path = "/content/drive/MyDrive/Colab Notebooks/runs/detect/train4/weights/best.pt"
model = YOLO(model_path)

# Load folder path
folder_path = "/content/drive/MyDrive/SFU Stuff/IAT 360/CV Project/Data/Sample Dataset/Tim Set/cleaned_v2"

# Get all image paths
image_paths = glob.glob(os.path.join(folder_path, "*.jpg")) + \
              glob.glob(os.path.join(folder_path, "*.png")) + \
              glob.glob(os.path.join(folder_path, "*.jpeg"))

print(f"Found {len(image_paths)} images in folder.")

# Loop through each image and predict
for img_path in image_paths:
    # Run detection
    results = model(img_path, conf=0.4, iou=0.45, show=False)
    annotated_img = results[0].plot()

    # Display prediction (Colab)
    print(f"\nPredictions for: {os.path.basename(img_path)}")
    cv2_imshow(annotated_img)

    # # Optionally save the annotated image (for presentation / report)
    # save_dir = "/content/drive/MyDrive/Colab Notebooks/runs/predict_custom/"
    # os.makedirs(save_dir, exist_ok=True)
    # out_path = os.path.join(save_dir, os.path.basename(img_path))
    # cv2.imwrite(out_path, annotated_img)

